In [1]:
from pathlib import Path
import pandas as pd
import torch
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

In [2]:
DATA_DIR = Path(r"E:\Shortcutscope\archive\waterbird_complete95_forest2water2")
METADATA_PATH = DATA_DIR / "metadata.csv"

IMAGE_SIZE = 128
BATCH_SIZE = 32

train_transform = transforms.Compose([
    transforms.RandomResizedCrop((IMAGE_SIZE, IMAGE_SIZE), scale=(0.4, 0.8)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4, hue=0.15),
    transforms.RandomGrayscale(p=0.3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.5, scale=(0.02, 0.2), value="random")
])




# 2. Clean evaluation transform for validation and test (no random augmentation)
eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


In [3]:
class WaterbirdsDataset(Dataset):

    def __init__(self, dataframe, data_dir, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.data_dir = Path(data_dir)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):

        row = self.dataframe.iloc[index]

        image_path = self.data_dir / row["img_filename"]

        image = Image.open(image_path).convert("RGB")

        label = int(row["y"])
        group = 2 * label + int(row["place"])

        if self.transform is not None:
            image = self.transform(image)

        return image, label, group


In [4]:

import pandas as pd
from pathlib import Path

DATA_DIR = Path(
    r"E:\Shortcutscope\archive\waterbird_complete95_forest2water2"
)
METADATA_PATH = DATA_DIR / "metadata.csv"

df = pd.read_csv(METADATA_PATH)

print("Metadata path:", METADATA_PATH.resolve())
print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())

print("\nFirst five rows:")
print(df.head().to_string())

print("\nSplit column data type:")
print(df["split"].dtype)

print("\nActual split values:")
print(df["split"].value_counts(dropna=False))

print("\nUnique split values:")
print(repr(df["split"].unique()))


Metadata path: E:\Shortcutscope\archive\waterbird_complete95_forest2water2\metadata.csv
Dataset shape: (11788, 6)
Columns: ['img_id', 'img_filename', 'y', 'split', 'place', 'place_filename']

First five rows:
   img_id                                                   img_filename  y  split  place                 place_filename
0       1  001.Black_footed_Albatross/Black_Footed_Albatross_0046_18.jpg  1      2      1          /o/ocean/00002178.jpg
1       2  001.Black_footed_Albatross/Black_Footed_Albatross_0009_34.jpg  1      0      1   /l/lake/natural/00000065.jpg
2       3  001.Black_footed_Albatross/Black_Footed_Albatross_0002_55.jpg  1      2      0  /b/bamboo_forest/00000131.jpg
3       4  001.Black_footed_Albatross/Black_Footed_Albatross_0074_59.jpg  1      0      1          /o/ocean/00001268.jpg
4       5  001.Black_footed_Albatross/Black_Footed_Albatross_0014_89.jpg  1      0      1          /o/ocean/00003147.jpg

Split column data type:
int64

Actual split values:
split
2    5

In [5]:

import pandas as pd

df = pd.read_csv(METADATA_PATH)

train_df = df.loc[df["split"] == 0].copy()
val_df = df.loc[df["split"] == 1].copy()
test_df = df.loc[df["split"] == 2].copy()

print("Training records:", len(train_df))
print("Validation records:", len(val_df))
print("Test records:", len(test_df))

assert len(train_df) == 4795
assert len(val_df) == 1199
assert len(test_df) == 5794

print("Official splits verified successfully!")


Training records: 4795
Validation records: 1199
Test records: 5794
Official splits verified successfully!


In [6]:
train_dataset = WaterbirdsDataset(train_df, DATA_DIR, transform=train_transform)
val_dataset = WaterbirdsDataset(val_df, DATA_DIR, transform=eval_transform)
test_dataset = WaterbirdsDataset(test_df, DATA_DIR, transform=eval_transform)


In [7]:
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

In [8]:
images, labels, groups = next(iter(train_loader))

print("\n--- Pipeline verification ---")
print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)
print("Group batch shape:", groups.shape)
print("Image data type:", images.dtype)
print("Labels:", labels[:10].tolist())
print("Groups:", groups[:10].tolist())

print("\nData pipeline loaded successfully!")


--- Pipeline verification ---
Image batch shape: torch.Size([32, 3, 128, 128])
Label batch shape: torch.Size([32])
Group batch shape: torch.Size([32])
Image data type: torch.float32
Labels: [0, 0, 1, 1, 0, 0, 0, 1, 0, 0]
Groups: [0, 0, 3, 2, 0, 0, 0, 3, 0, 0]

Data pipeline loaded successfully!
